# Cell 1: Import everything:


In [1]:
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader

# Data handeling
from src.data.dataset import MLMDataset, ClassificationDataset
from src.data.data_tools import filter_taxonomy, fasta2pandas

#Vocab shit
from src.utils.vocab import Vocabulary, KmerVocabConstructor

# Preprocessing
from src.preprocessing.augmentation import SequenceModifier, IdentityStrategy, BaseStrategy
from src.preprocessing.tokenization import KmerStrategy
from src.preprocessing.padding import PEndStrategy
from src.preprocessing.truncation import TEndStrategy
from src.preprocessing.preprocessor import Preprocessor

# Model things
from src.model.backbone import Bertax, ModularBertax
from src.model.encoders import LabelEncoder
from src.model.heads import MLMHead, SingleClassHead

# Training things
from src.train.trainers import MLMtrainer, ClassificationTrainer

In [2]:


CONFIG = {
    "FILE_PATH": "src/data/raw.fasta",
    "SAVE_PATH": "pretrained_model.pt",
    "n_test": 10,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": 3,
    "optimal_length": 200,

    # Training parameters
    "num_epochs": 1,
    "masking_percentage": 0.05,
    "batch_size": 128,
    "small_set": False,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1,

    #Classification 
    "target_label": "phylum"
}


In [3]:

# Set up vocabulary
constructor = KmerVocabConstructor(k=CONFIG["k"], alphabet=CONFIG["alphabet"])
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])
vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessors
sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=CONFIG["modification_probability"]
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=0
)

tokenization_strategy = KmerStrategy(
    k=CONFIG["k"],
    padding_alphabet=CONFIG["alphabet"]
)

padding_strategy = PEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

truncation_strategy = TEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

preprocessor_train = Preprocessor(
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

preprocessor_val = Preprocessor(
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)



In [4]:

##################################################################
## Data preparation ##############################################
##################################################################

all_data = fasta2pandas(CONFIG["FILE_PATH"])

#tmp: to use a smaller set for test if code compiles - nothing to be used in production
if CONFIG["small_set"]:
    all_data = all_data[:CONFIG["n_test"]]

# filter data for finetuning
filtered_data = filter_taxonomy(
    df = all_data,
    startAt ='phylum',
    endAt = 'phylum',
    phylumCertainty=True
    )

# 
num_classes = len(list(set(filtered_data[CONFIG["target_label"]])))

# encodes label for classification
label_encoder = LabelEncoder(filtered_data[CONFIG["target_label"]])

# Pretraining datasplit based on unsplit data: 
pretrain_sequences, preval_sequences = train_test_split(
    all_data["sequence"],
    test_size = 0.1,
    random_state = 42
)

# Finetune datasplit based on filtered data:
finetrain_data, fineval_data = train_test_split(
    filtered_data,
    test_size = 0.1,
    random_state = 69
    )

print(f"Number of pre-training sequences: {len(pretrain_sequences)}")
print(f"Number of validation sequences: {len(preval_sequences)}")


Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing row 10000...
Processing row 20000...
Processing row 30000...
Processing row 40000...
Processing row 50000...
Processing row 60000...
Processing row 70000...
Processing row 80000...
Processing row 90000...
Processing complete.
Number of pre-training sequences: 83962
Number of validation sequences: 9330


In [5]:
##################################################################
## Setup Datasets ################################################
##################################################################

# Pretraining datasets
pretrain_dataset = MLMDataset(
    df = pretrain_sequences,
    preprocessor = preprocessor_train,
    masking_percentage = CONFIG["masking_percentage"]
)

preval_dataset = MLMDataset(
    df = preval_sequences,
    preprocessor = preprocessor_val,
    masking_percentage = CONFIG["masking_percentage"]
)

# Finetuning datasets
finetrain_dataset = ClassificationDataset(
    df = finetrain_data,
    preprocessor = preprocessor_train,
    label_encoder = label_encoder,
    target_column = "phylum"
    )

fineval_dataset = ClassificationDataset(
    df = fineval_data,
    preprocessor = preprocessor_val,
    label_encoder = label_encoder,
    target_column = "phylum"
    )

##################################################################
## Setup Dataloader ##############################################
##################################################################

pretrain_loader = DataLoader(
    dataset = pretrain_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

preval_loader = DataLoader(
    dataset = preval_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

#data loaders
finetrain_loader = DataLoader(
    dataset=finetrain_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )

fineval_loader = DataLoader(
    dataset=fineval_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )



In [6]:
encoder_config = BertConfig(
    vocab_size = len(vocab),
    hidden_size = CONFIG["hidden_size"],
    num_hidden_layers = CONFIG["num_layers"],
    num_attention_heads = CONFIG["num_attention_heads"],
    intermediate_size = CONFIG["intermediate_size"],
    max_position_embeddings = CONFIG["optimal_length"] + 2,
    hidden_dropout_prob = CONFIG["dropout_rate"],
    attention_probs_dropout_prob = CONFIG["dropout_rate"]
)

encoder = BertModel(encoder_config)

mlm_head = MLMHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = len(vocab),
    dropout_rate = CONFIG["mlm_dropout_rate"]
)

classification_head = SingleClassHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = num_classes,
    dropout_rate = CONFIG["mlm_dropout_rate"]
)

model = ModularBertax(
    encoder = encoder,
    mlm_head = mlm_head,
    classification_head = classification_head
)

mlm_trainer = MLMtrainer(
    model = model,
    train_loader = pretrain_loader,
    val_loader = preval_loader
)

classification_trainer = ClassificationTrainer(
    model,
    train_loader = finetrain_loader,
    val_loader = fineval_loader
    )

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
model.preTrainMode()
mlm_trainer.train(CONFIG["num_epochs"])
torch.save(model.state_dict(), CONFIG["SAVE_PATH"])
print(f"Model's state dictionary saved to {CONFIG['SAVE_PATH']}")

model.classifyMode()
classification_trainer.train(CONFIG["num_epochs"])

KeyboardInterrupt: 

In [ ]:
print(model.named_parameters())

<generator object Module.named_parameters at 0x000001B81C3AEB40>


# Eval model with single input: 

In [58]:
model.classifyMode()
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 4) Get a single item from your dataset
item = finetrain_dataset.__getitem__(idx=5)
input_ids = item["input_ids"].unsqueeze(0).to(device)       # [1, seq_len]
attention_mask = item["attention_mask"].unsqueeze(0).to(device)
label = item["label"]
encoded_label = item["encoded_label"]
# 5) Inference
with torch.no_grad():
    logits = model(input_ids, attention_mask)  # shape: [1, num_classes]
    predicted_class = logits.argmax(dim=-1).item()

print(input_ids)
print(attention_mask)
print(label)
print(encoded_label)
print("Logits:", logits)
print("Predicted class index:", predicted_class)

tensor([[11,  8,  5, 68, 67, 15, 19, 67, 35, 50, 67, 60, 66, 47, 19, 47, 23, 51,
         22, 60, 57,  8, 20, 28, 67, 10, 25, 36, 17, 25, 10, 19, 61, 11, 35, 64,
         16, 43, 41, 68, 67, 66, 21, 28, 19, 66, 23, 57, 68, 44, 30,  6, 37, 64,
         64, 59,  8, 52,  8, 36,  6,  8,  8,  8, 29, 21, 36, 33,  9, 19, 40, 34,
         67, 44, 66, 41, 59, 19,  5, 11, 23, 29,  8, 43, 17, 32, 15, 61, 20, 41,
         13, 20, 27, 61, 18, 18, 37, 60, 67,  6, 41, 28, 62, 42, 28, 63, 56, 58,
         39, 47, 24, 42, 64, 67, 16, 50, 20, 49, 33, 60, 21, 36, 67, 12, 67, 37,
         52, 25, 48, 31, 14, 67, 40, 52, 45, 48, 52, 44, 46, 48, 36, 50, 48, 23,
         34, 28, 37,  7, 56, 55, 29,  7, 41,  8, 41, 27, 18, 26, 41, 60, 24, 44,
         60, 46, 20, 40, 16, 64, 33, 62, 36, 64, 43, 44, 37, 19, 36, 56, 47, 20,
         44, 39, 61, 28, 68, 68, 64, 25, 23, 19, 59, 23, 38, 10, 44,  0,  0,  0,
          0,  0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         